# IESO Coincident Peak Prediction — Operational Design & Deployment Architecture

This notebook demonstrates a live prediction workflow using the trained model
with current IESO demand data and weather forecasts, and documents the
deployment architecture for operational use.

**Three-phase daily workflow:**
1. Morning forecast (6–7 AM): weather-based demand prediction and risk classification
2. Intraday monitoring (noon–8 PM): real-time demand trajectory tracking
3. Post-day review: accuracy logging and threshold update

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import requests
import warnings
from pathlib import Path
from datetime import datetime, timedelta, date

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

PROJECT_ROOT = Path(r'C:/wamp64/www/Spec_Driven_Dev_Website')
DATA_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'data'
MODEL_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'models'

print('Libraries loaded successfully')

In [ ]:
# Load model and data
model_artifact = joblib.load(MODEL_DIR / 'ieso_peak_model.joblib')
model = model_artifact['model']
FEATURE_COLS = model_artifact['feature_columns']

features = pd.read_parquet(DATA_DIR / 'ieso_features_daily.parquet')
features['Date'] = pd.to_datetime(features['Date'])
hourly = pd.read_parquet(DATA_DIR / 'ieso_hourly_with_features.parquet')
hourly['Date'] = pd.to_datetime(hourly['Date'])
peaks = pd.read_parquet(DATA_DIR / 'ieso_peak_labels.parquet')
peaks['date'] = pd.to_datetime(peaks['date'])

print(f'Model loaded: test RMSE = {model_artifact["test_rmse"]:.1f} MW')
print(f'Features: {FEATURE_COLS}')

## Live Prediction Demo

Fetch current weather forecast from Open-Meteo and most recent demand data
from the local dataset, then run the trained model to generate today's
risk assessment.

In [ ]:
def fetch_weather_forecast(lat=43.65, lon=-79.38, days=7):
    """Fetch weather forecast from Open-Meteo."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'hourly': 'temperature_2m,relative_humidity_2m,dewpoint_2m',
        'forecast_days': days,
        'timezone': 'America/Toronto'
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        hourly_data = data['hourly']
        df = pd.DataFrame({
            'datetime': pd.to_datetime(hourly_data['time']),
            'temperature_c': hourly_data['temperature_2m'],
            'relative_humidity': hourly_data['relative_humidity_2m'],
            'dewpoint_c': hourly_data['dewpoint_2m'],
        })
        df['date'] = df['datetime'].dt.date
        return df
    except Exception as e:
        print(f'Weather forecast fetch failed: {e}')
        return None

def compute_humidex(temp_c, dewpoint_c):
    e = 6.11 * np.exp(5417.7530 * (1.0/273.16 - 1.0/(273.15 + dewpoint_c)))
    h = temp_c + (5.0/9.0) * (e - 10.0)
    return np.where(temp_c >= 20, h, temp_c)

# Fetch forecast
print('Fetching 7-day weather forecast for Toronto...')
forecast = fetch_weather_forecast()

if forecast is not None:
    print(f'Forecast retrieved: {len(forecast)} hourly records')
    print(f'Date range: {forecast["date"].min()} to {forecast["date"].max()}')
    
    # Compute daily weather features from forecast
    forecast['humidex'] = compute_humidex(forecast['temperature_c'].values, 
                                          forecast['dewpoint_c'].values)
    forecast['cdh'] = np.maximum(0, forecast['temperature_c'] - 18.0)
    
    daily_forecast = forecast.groupby('date').agg({
        'temperature_c': ['max', 'mean'],
        'humidex': 'max',
        'dewpoint_c': 'mean',
        'relative_humidity': 'mean',
        'cdh': 'sum',
    }).reset_index()
    daily_forecast.columns = ['date', 'daily_max_temp', 'daily_mean_temp',
                               'daily_max_humidex', 'daily_mean_dewpoint',
                               'daily_mean_rh', 'daily_cdh']
    
    print('\n7-Day Forecast Summary:')
    print(daily_forecast[['date', 'daily_max_temp', 'daily_max_humidex', 
                          'daily_cdh']].to_string(index=False))
else:
    print('Using most recent historical data as fallback')

In [ ]:
def predict_daily_risk(forecast_row, recent_features, model, feature_cols):
    """Generate risk prediction for a single day."""
    # Build feature vector from forecast + recent demand history
    feature_dict = {}
    
    # Weather features from forecast
    feature_dict['daily_max_temp'] = forecast_row.get('daily_max_temp', 20)
    feature_dict['daily_mean_temp'] = forecast_row.get('daily_mean_temp', 15)
    feature_dict['daily_max_humidex'] = forecast_row.get('daily_max_humidex', 20)
    feature_dict['daily_cdh'] = forecast_row.get('daily_cdh', 0)
    feature_dict['daily_mean_rh'] = forecast_row.get('daily_mean_rh', 60)
    feature_dict['daily_mean_dewpoint'] = forecast_row.get('daily_mean_dewpoint', 10)
    
    # Rolling averages (use recent history + forecast)
    if recent_features is not None and len(recent_features) >= 3:
        recent = recent_features.tail(3)
        feature_dict['temp_3day_avg'] = recent['daily_max_temp'].mean()
        feature_dict['cdh_3day_avg'] = recent['daily_cdh'].mean()
    else:
        feature_dict['temp_3day_avg'] = feature_dict['daily_max_temp']
        feature_dict['cdh_3day_avg'] = feature_dict['daily_cdh']
    
    # Calendar features
    pred_date = pd.Timestamp(forecast_row['date'])
    feature_dict['month'] = pred_date.month
    feature_dict['day_of_week'] = pred_date.dayofweek
    feature_dict['is_business_day'] = 1 if pred_date.dayofweek < 5 else 0
    feature_dict['day_of_year'] = pred_date.dayofyear
    
    # Demand momentum from recent history
    if recent_features is not None and len(recent_features) > 0:
        feature_dict['prev_day_max_demand'] = recent_features['daily_max_demand'].iloc[-1]
        feature_dict['rolling_7d_max_demand'] = recent_features.tail(7)['daily_max_demand'].max()
        feature_dict['rolling_7d_mean_demand'] = recent_features.tail(7)['daily_max_demand'].mean()
    else:
        feature_dict['prev_day_max_demand'] = 16000
        feature_dict['rolling_7d_max_demand'] = 18000
        feature_dict['rolling_7d_mean_demand'] = 16000
    
    # Peak context
    if recent_features is not None and 'current_5th_peak' in recent_features.columns:
        feature_dict['current_5th_peak'] = recent_features['current_5th_peak'].iloc[-1]
        feature_dict['max_demand_so_far'] = recent_features['max_demand_so_far'].iloc[-1]
    else:
        feature_dict['current_5th_peak'] = 20000
        feature_dict['max_demand_so_far'] = 20000
    
    # Create feature vector
    X = pd.DataFrame([feature_dict])[feature_cols]
    
    # Predict
    predicted_demand = model.predict(X)[0]
    
    # Classify risk
    threshold = feature_dict['current_5th_peak']
    buffer = 500
    diff = predicted_demand - threshold
    
    if diff > buffer:
        risk = 'RED'
    elif abs(diff) <= buffer:
        risk = 'YELLOW'
    else:
        risk = 'GREEN'
    
    # Predict peak window
    max_temp = feature_dict['daily_max_temp']
    if max_temp > 33:
        window = 'HE 17-19 (4-7 PM)'
    elif max_temp > 28:
        window = 'HE 16-18 (3-6 PM)'
    else:
        window = 'HE 15-17 (2-5 PM)'
    
    return {
        'date': forecast_row['date'],
        'predicted_max_demand': predicted_demand,
        'threshold': threshold,
        'risk_level': risk,
        'predicted_window': window,
        'max_temp_forecast': max_temp,
    }

# Get recent features from the latest available data
recent_features = features.tail(30).copy()

# Generate predictions for 7-day outlook
if forecast is not None:
    outlook = []
    for _, row in daily_forecast.iterrows():
        pred = predict_daily_risk(row.to_dict(), recent_features, model, FEATURE_COLS)
        outlook.append(pred)
    
    outlook_df = pd.DataFrame(outlook)
    
    # Color-coded display
    print('\n' + '='*70)
    print('  7-DAY PEAK RISK OUTLOOK')
    print('='*70)
    for _, row in outlook_df.iterrows():
        risk_icon = {'RED': '!!!', 'YELLOW': ' ! ', 'GREEN': '   '}[row['risk_level']]
        print(f"  {row['date']}  [{risk_icon}] {row['risk_level']:6s}  "
              f"Pred: {row['predicted_max_demand']:,.0f} MW  "
              f"Temp: {row['max_temp_forecast']:.0f}°C  "
              f"Window: {row['predicted_window']}")
    print('='*70)
else:
    print('Forecast unavailable — using historical demo')
    outlook_df = pd.DataFrame()

## Intraday Tracking Visualization

Show how accumulating real-time demand data through the day narrows
uncertainty by comparing the current trajectory against historical
peak-day demand envelopes.

In [ ]:
# Build historical peak-day demand profiles
top5 = peaks[peaks['rank'] <= 5].copy()
peak_dates = top5['date'].dt.date.unique()

# Get hourly profiles for all peak days
peak_profiles = []
for pd_date in peak_dates:
    day_data = hourly[hourly['Date'].dt.date == pd_date]
    if len(day_data) >= 20:  # Need sufficient hours
        profile = day_data[['Hour', 'Ontario Demand']].copy()
        peak_profiles.append(profile)

if peak_profiles:
    # Compute envelope (10th and 90th percentile at each hour)
    all_profiles = pd.concat(peak_profiles)
    envelope = all_profiles.groupby('Hour')['Ontario Demand'].agg(
        p10=lambda x: np.percentile(x, 10),
        p25=lambda x: np.percentile(x, 25),
        p50='median',
        p75=lambda x: np.percentile(x, 75),
        p90=lambda x: np.percentile(x, 90),
    ).reset_index()

# Get a recent day's actual trajectory for comparison
recent_day = hourly[hourly['Date'] == hourly['Date'].max()]

# Also get a typical non-peak day for contrast
non_peak_dates = hourly[
    (~hourly['Date'].dt.date.isin(peak_dates)) & 
    (hourly['Date'].dt.month.isin([6, 7, 8]))
]['Date'].unique()
if len(non_peak_dates) > 0:
    typical_day = hourly[hourly['Date'] == non_peak_dates[-10]]

fig, ax = plt.subplots(figsize=(14, 7))

# Peak day envelope
if peak_profiles:
    ax.fill_between(envelope['Hour'], envelope['p10'], envelope['p90'],
                    alpha=0.15, color='#d32f2f', label='Peak days (10th–90th pctl)')
    ax.fill_between(envelope['Hour'], envelope['p25'], envelope['p75'],
                    alpha=0.25, color='#d32f2f', label='Peak days (25th–75th pctl)')
    ax.plot(envelope['Hour'], envelope['p50'], '--', color='#d32f2f', 
            linewidth=2, label='Peak day median')

# Current day trajectory
if len(recent_day) > 0:
    ax.plot(recent_day['Hour'], recent_day['Ontario Demand'], 'o-', 
            color='#1565C0', linewidth=2, markersize=4,
            label=f'Latest data ({recent_day["Date"].iloc[0].strftime("%Y-%m-%d")})')

# Typical non-peak day
if len(non_peak_dates) > 0 and len(typical_day) > 0:
    ax.plot(typical_day['Hour'], typical_day['Ontario Demand'], '--', 
            color='#9E9E9E', linewidth=1.5, alpha=0.7,
            label='Typical summer day (non-peak)')

# Mark the peak window
ax.axvspan(15, 19, alpha=0.08, color='orange')
ax.text(17, ax.get_ylim()[1] * 0.98, 'Peak\nWindow', 
        ha='center', va='top', fontsize=10, color='#FF9800', fontweight='bold')

ax.set_xlabel('Hour Ending (EST)')
ax.set_ylabel('Ontario Demand (MW)')
ax.set_title('Intraday Demand Tracking — Current Day vs. Historical Peak Envelopes')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_xlim(1, 24)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig(DATA_DIR / 'intraday_tracking.png', dpi=150, bbox_inches='tight')
plt.show()

## 7-Day Forecast Calendar

In [ ]:
if len(outlook_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 3))
    
    risk_colors = {'RED': '#FFCDD2', 'YELLOW': '#FFF9C4', 'GREEN': '#C8E6C9'}
    risk_text_colors = {'RED': '#B71C1C', 'YELLOW': '#F57F17', 'GREEN': '#2E7D32'}
    
    for i, (_, row) in enumerate(outlook_df.iterrows()):
        color = risk_colors[row['risk_level']]
        text_color = risk_text_colors[row['risk_level']]
        
        rect = plt.Rectangle((i, 0), 0.9, 1, facecolor=color, 
                              edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        
        # Date
        ax.text(i + 0.45, 0.85, str(row['date']), ha='center', va='top',
                fontsize=8, fontweight='bold')
        # Risk level
        ax.text(i + 0.45, 0.55, row['risk_level'], ha='center', va='center',
                fontsize=14, fontweight='bold', color=text_color)
        # Predicted demand
        ax.text(i + 0.45, 0.3, f"{row['predicted_max_demand']:,.0f} MW", 
                ha='center', va='center', fontsize=8)
        # Temperature
        ax.text(i + 0.45, 0.12, f"{row['max_temp_forecast']:.0f}°C",
                ha='center', va='center', fontsize=8, color='#555')
    
    ax.set_xlim(-0.2, len(outlook_df) + 0.2)
    ax.set_ylim(-0.1, 1.1)
    ax.axis('off')
    ax.set_title('7-Day Peak Risk Outlook', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'forecast_outlook.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No forecast data available for calendar display')

## API Design Specification

REST API endpoints for an operational deployment of the prediction service.

| Endpoint | Method | Description | Response |
|----------|--------|-------------|----------|
| `/predict/today` | GET | Morning forecast with risk level | `{date, predicted_max_mw, risk_level, window, confidence}` |
| `/predict/realtime` | GET | Intraday updated probability using actual demand trajectory | `{date, hour, current_demand_mw, projected_max_mw, risk_level, trajectory_percentile}` |
| `/predict/outlook` | GET | 7-day forecast outlook | `[{date, risk_level, predicted_max_mw, max_temp_c}]` |
| `/status/peaks` | GET | Current top-5 peaks and displacement threshold | `{base_period, peaks: [{rank, date, hour, demand_mw}], threshold_mw}` |
| `/status/model` | GET | Model metadata and performance | `{version, trained_on, test_rmse, last_retrained}` |
| `/history/backtest` | GET | Walk-forward backtest results | `[{base_period, rmse, precision, recall, f1}]` |
| `/history/alerts` | GET | Alert history for current base period | `[{date, alert_level, predicted_mw, actual_mw, was_peak}]` |

## Deployment Architecture

```
                    IESO Coincident Peak Prediction — System Architecture
                    =====================================================

    ┌─────────────────────┐     ┌──────────────────────┐
    │   IESO Public APIs  │     │  Open-Meteo Weather  │
    │                     │     │                      │
    │  • Hourly Demand    │     │  • 7-Day Forecast    │
    │  • ICI Demand       │     │  • Historical Archive│
    │  • Peak Tracker     │     │                      │
    └────────┬────────────┘     └──────────┬───────────┘
             │                              │
             ▼                              ▼
    ┌─────────────────────────────────────────────────┐
    │              Python Ingestion Layer             │
    │                                                 │
    │  • Fetch & validate data                        │
    │  • Timestamp alignment (HE→datetime)            │
    │  • Gap detection & handling                     │
    │  • Cron: 6 AM daily + hourly noon–8 PM          │
    └────────────────────┬────────────────────────────┘
                         │
                         ▼
    ┌─────────────────────────────────────────────────┐
    │               Feature Engine                    │
    │                                                 │
    │  • Weather → humidex, CDH, rolling averages     │
    │  • Demand → lagged, rolling, momentum           │
    │  • Peak context → threshold tracker             │
    │  • Calendar → holidays, business day flags      │
    └────────────────────┬────────────────────────────┘
                         │
                         ▼
    ┌─────────────────────────────────────────────────┐
    │           XGBoost Prediction Model              │
    │                                                 │
    │  • Daily max demand regression                  │
    │  • RED / YELLOW / GREEN classification           │
    │  • 3-hour peak window estimation                │
    │  • Retrained annually (May 1)                   │
    └────────────────────┬────────────────────────────┘
                         │
                         ▼
    ┌─────────────────────────────────────────────────┐
    │            Flask/FastAPI REST Service            │
    │                                                 │
    │  GET /predict/today    → morning forecast       │
    │  GET /predict/realtime → intraday update        │
    │  GET /predict/outlook  → 7-day outlook          │
    │  GET /status/peaks     → current threshold      │
    └────────────────────┬────────────────────────────┘
                         │
            ┌────────────┼────────────┐
            ▼            ▼            ▼
    ┌──────────┐  ┌──────────┐  ┌──────────┐
    │  Email   │  │   SMS    │  │  Slack   │
    │ (SMTP)   │  │ (Twilio) │  │ Webhook  │
    └──────────┘  └──────────┘  └──────────┘
```

## Model Retraining Schedule

The model should be retrained annually to incorporate the latest base period's data.

| Step | Timing | Action |
|------|--------|--------|
| 1 | May 1 | Base period closes. IESO publishes final top-5 peaks. |
| 2 | May 1–7 | Download completed base period data. Add to training set. |
| 3 | May 7–14 | Retrain XGBoost on full historical dataset (2010–latest). |
| 4 | May 14–21 | Validate on previous base period. Compare RMSE and recall to prior model. |
| 5 | June 1 | Deploy new model for upcoming peak season. |
| 6 | Jun–Sep | Monitor prediction accuracy. Flag drift if RMSE > 1.5× historical average. |
| 7 | Ongoing | Track structural demand shifts (EV adoption, new industrial loads, BTM solar). |

## Monitoring & Drift Detection

Key metrics to track during the peak season:

In [ ]:
# Simulate monitoring dashboard with historical data
# Show prediction accuracy over the most recent summer
bp2024 = features[(features['base_period'] == 2024) & 
                   features['month'].isin([6, 7, 8])].copy()

if len(bp2024) > 0 and len(bp2024.dropna(subset=FEATURE_COLS)) > 0:
    bp2024_clean = bp2024.dropna(subset=FEATURE_COLS)
    bp2024_clean['predicted'] = model.predict(bp2024_clean[FEATURE_COLS])
    bp2024_clean['error'] = bp2024_clean['predicted'] - bp2024_clean['daily_max_demand']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Actual vs Predicted scatter
    axes[0, 0].scatter(bp2024_clean['daily_max_demand'], bp2024_clean['predicted'],
                       alpha=0.5, s=20, color='#1565C0')
    lims = [bp2024_clean[['daily_max_demand', 'predicted']].min().min(),
            bp2024_clean[['daily_max_demand', 'predicted']].max().max()]
    axes[0, 0].plot(lims, lims, '--', color='gray')
    axes[0, 0].set_xlabel('Actual Max Demand (MW)')
    axes[0, 0].set_ylabel('Predicted Max Demand (MW)')
    axes[0, 0].set_title('Actual vs. Predicted')
    
    # Error distribution
    axes[0, 1].hist(bp2024_clean['error'], bins=30, color='#4CAF50', alpha=0.7,
                    edgecolor='white')
    axes[0, 1].axvline(x=0, color='black', linestyle='--')
    axes[0, 1].set_xlabel('Prediction Error (MW)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title(f'Error Distribution (RMSE={bp2024_clean["error"].std():.0f} MW)')
    
    # Cumulative error over time
    axes[1, 0].plot(bp2024_clean['Date'], bp2024_clean['error'].cumsum(),
                    color='#FF9800', linewidth=1.5)
    axes[1, 0].axhline(y=0, color='gray', linestyle='--')
    axes[1, 0].set_xlabel('Date')
    axes[1, 0].set_ylabel('Cumulative Error (MW)')
    axes[1, 0].set_title('Cumulative Prediction Bias')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Rolling RMSE (7-day window)
    rolling_rmse = bp2024_clean['error'].rolling(7).apply(
        lambda x: np.sqrt(np.mean(x**2))
    )
    axes[1, 1].plot(bp2024_clean['Date'], rolling_rmse, color='#d32f2f', linewidth=1.5)
    overall_rmse = np.sqrt(np.mean(bp2024_clean['error']**2))
    axes[1, 1].axhline(y=overall_rmse, color='gray', linestyle='--', 
                        label=f'Overall RMSE: {overall_rmse:.0f} MW')
    axes[1, 1].axhline(y=overall_rmse * 1.5, color='#d32f2f', linestyle=':', 
                        label=f'Drift threshold: {overall_rmse*1.5:.0f} MW')
    axes[1, 1].set_xlabel('Date')
    axes[1, 1].set_ylabel('7-Day Rolling RMSE (MW)')
    axes[1, 1].set_title('Model Drift Monitoring')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.suptitle('Model Performance Dashboard — Summer 2024', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'monitoring_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insufficient 2024 summer data for monitoring dashboard')

print('\n=== Notebook 5 complete ===')